# Exercise 11 : Integration Case

Answer the question of the slides : how many road deaths per million inhabitants did 33 European
countries record in 2024, per age group? Steps 1–12 follow the twelve steps on the slides ; try
each step yourself before you look back. Step 13 audits a flawed analysis.

Read `datasets/eurostat-road-safety-README.md` first. The files are unchanged Eurostat extracts,
retrieved on 10 October 2026 : the road deaths of 2023 and 2024 (`tran_sf_roadus`) and the
population on 1 January 2024 (`demo_pjan`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA = Path.cwd().parents[1] / 'datasets'

## Step 1 : Raw files

Read the death file without options. Look at its shape, its first rows and the distinct values of
`unit`, `age` and `geo`. Which rows must you leave out to count each death once? Look at the
population file in the same way : which age labels does it use?

In [ ]:
raw = pd.read_csv(DATA / 'eurostat-road-deaths-2023-2024.csv')
print(raw.shape)
print(raw[['unit', 'sex', 'age', 'pers_cat', 'geo', 'TIME_PERIOD', 'OBS_VALUE']].head(3))
print(raw['unit'].unique().tolist())
print(raw['age'].unique().tolist())
print(raw['geo'].nunique(), 'EU27_2020' in set(raw['geo']))

## Step 2 : Constants

The notebook defines `AGE_CODES`, which maps each Eurostat age code to a label, `LABELS` and `BINS`,
as on the slide. Print `LABELS`. `BINS` has seven edges and makes six population bands : which
label has no band?

In [ ]:
AGE_CODES = {'Y_LT15': 'Under 15', 'Y15-17': '15-17', 'Y18-24': '18-24',
             'Y25-49': '25-49', 'Y50-64': '50-64', 'Y_GE65': '65+',
             'UNK': 'Unknown age'}
LABELS = list(AGE_CODES.values())
BINS = [0, 15, 18, 25, 50, 65, float('inf')]

In [ ]:
print(LABELS)
print(len(BINS) - 1, len(LABELS))

## Step 3 : Typed load

Read both files with `usecols` and with `Int64` for `TIME_PERIOD` and `OBS_VALUE`. Rename
`TIME_PERIOD`, `OBS_VALUE` and `OBS_FLAG` to `year`, `value` and `flag`. Print the row counts, the
number of empty values and the population flags.

In [ ]:
names = {'TIME_PERIOD': 'year', 'OBS_VALUE': 'value', 'OBS_FLAG': 'flag'}
deaths = pd.read_csv(
    DATA / 'eurostat-road-deaths-2023-2024.csv',
    usecols=['unit', 'sex', 'age', 'pers_cat', 'geo', 'TIME_PERIOD', 'OBS_VALUE'],
    dtype={'TIME_PERIOD': 'Int64', 'OBS_VALUE': 'Int64'},
).rename(columns=names)
population = pd.read_csv(
    DATA / 'eurostat-population-2024.csv',
    usecols=['sex', 'age', 'geo', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG'],
    dtype={'TIME_PERIOD': 'Int64', 'OBS_VALUE': 'Int64'},
).rename(columns=names)
print(deaths.shape, population.shape)
print(deaths['value'].isna().sum(), population['value'].isna().sum())
print(population['flag'].value_counts())

## Step 4 : Death counts

Keep the rows with `unit == 'NR'`, `pers_cat == 'TOTAL'`, year 2024 and `sex == 'T'`, without
`EU27_2020`. Store the sorted list of countries as `COUNTRIES`, each country's `TOTAL` row as
`totals` and the remaining age rows as `counts`. Check : 33 countries, 219 age rows and 20,875
deaths.

In [ ]:
counts = deaths.loc[deaths['unit'].eq('NR') & deaths['pers_cat'].eq('TOTAL')
                    & deaths['year'].eq(2024) & deaths['sex'].eq('T')
                    & deaths['geo'].ne('EU27_2020')]
COUNTRIES = sorted(counts['geo'].unique())
totals = counts.loc[counts['age'].eq('TOTAL')].set_index('geo')['value']
counts = counts.loc[counts['age'].ne('TOTAL')]
print(len(COUNTRIES), len(counts), totals.sum())

## Step 5 : Age labels

Map the age codes to labels with `AGE_CODES` and keep the columns `geo`, `age_group` and `deaths`.
Check that no label is missing.

In [ ]:
counts = counts.assign(age_group=counts['age'].map(AGE_CODES))
counts = counts[['geo', 'age_group', 'value']].rename(columns={'value': 'deaths'})
print(counts.head(4))
print(counts['age_group'].isna().sum())

## Step 6 : Population bands

Keep the population for `sex == 'T'` without `EU27_2020`. Leave out `TOTAL` and `UNK`, turn `Y_LT1`
into 0 and `Y_OPEN` into 100, and band the ages with `pd.cut(..., right=False)`. Sum the result to
one row per country and band, in a column `population`.

In [ ]:
people = population.loc[population['sex'].eq('T') & population['geo'].ne('EU27_2020')]
ages = people.loc[~people['age'].isin(['TOTAL', 'UNK'])]
years = ages['age'].replace({'Y_LT1': 'Y0', 'Y_OPEN': 'Y100'}).str[1:].astype(int)
ages = ages.assign(age_group=pd.cut(years, BINS, right=False, labels=LABELS[:-1]))
bands = (ages.groupby(['geo', 'age_group'], observed=True)['value'].sum()
         .reset_index(name='population'))
bands['age_group'] = bands['age_group'].astype(str)
print(len(ages) // len(COUNTRIES), 'ages per country ->', len(bands) // len(COUNTRIES), 'bands')
print(bands.head(3))

## Step 7 : Band check

Check that the bands of each country add up to its published `TOTAL` population. Check one cell by
hand : the under-15 population of the Netherlands is 2,714,040.

In [ ]:
published = people.loc[people['age'].eq('TOTAL')].set_index('geo')['value']
banded = bands.groupby('geo')['population'].sum()
print((banded == published).all())
print(bands.loc[bands['geo'].eq('NL') & bands['age_group'].eq('Under 15'), 'population'].iloc[0])

## Step 8 : Grid

Build the grid of 33 countries and 7 labels with `pd.MultiIndex.from_product()`. Left-merge the
bands and then the counts onto it, with `validate='one_to_one'` and `indicator=True`. Check that the
grid has 231 cells and that the deaths of each country add up to its `totals`.

In [ ]:
cells = pd.MultiIndex.from_product([COUNTRIES, LABELS], names=['geo', 'age_group'])
table = cells.to_frame(index=False)
table = table.merge(bands, how='left', validate='one_to_one')
table = table.merge(counts, how='left', validate='one_to_one', indicator=True)
print(len(table), table['_merge'].value_counts().to_dict())
recorded = table.groupby('geo')['deaths'].sum().loc[totals.index]
print((recorded == totals).all(), recorded.sum())

## Step 9 : Missing counts

List the cells without a death count. Which countries are they? Why must these cells stay missing?

In [ ]:
missing = table.loc[table['deaths'].isna()]
print(missing['geo'].value_counts().to_dict())
print(table.loc[table['geo'].isin(['LI', 'ME']) & table['age_group'].eq('Unknown age'),
                ['geo', 'deaths']])

## Step 10 : Rates

Build the summary per age group with the columns `countries`, `reported`, `missing`, `deaths`,
`population` and `per_million`. `population` counts only the countries that report the group.
Which age group has the highest rate, and which has the most deaths?

In [ ]:
table['reported_population'] = table['population'].where(table['deaths'].notna())
grouped = table.groupby('age_group', sort=False)
summary = pd.DataFrame({
    'countries': grouped.size(),
    'reported': grouped['deaths'].count(),
    'deaths': grouped['deaths'].sum(),
    'population': grouped['reported_population'].sum(min_count=1),
})
summary['missing'] = summary['countries'] - summary['reported']
summary['per_million'] = (summary['deaths'] / summary['population'] * 1_000_000).round(1)
summary = summary[['countries', 'reported', 'missing', 'deaths', 'population', 'per_million']]
print(summary)
print(summary['deaths'].sum())

## Step 11 : Comparisons

Compare your country rates with Eurostat's own rates, `unit == 'P_MHAB'` : how many of the 186
agree after rounding? Compare your deaths with `datasets/road-deaths-expected-summary.csv`, which
was computed separately with SQL.

In [ ]:
rates = deaths.loc[deaths['unit'].eq('P_MHAB') & deaths['sex'].eq('T')
                   & deaths['pers_cat'].eq('TOTAL') & deaths['year'].eq(2024)
                   & deaths['age'].isin(list(AGE_CODES)[:-1])]
rates = rates.assign(age_group=rates['age'].map(AGE_CODES))
mine = table.assign(rate=table['deaths'] / table['population'] * 1_000_000)
both = mine.merge(rates, on=['geo', 'age_group'], validate='one_to_one')
print((both['rate'].round() == both['value']).sum(), 'of', len(both), 'country rates match')

expected = pd.read_csv(DATA / 'road-deaths-expected-summary.csv').set_index('age_group')
print((expected['deaths'] == summary['deaths']).all(),
      (expected['per_million'].round(1) == summary['per_million']).all())

## Step 12 : Chart

Draw one bar per age group with the rate per million. Label each bar with its deaths and
population, and name what the chart leaves out in the axis label. Save the figure as
`road-deaths-by-age.png` before you show it.

In [ ]:
shown = summary.loc[summary['per_million'].notna()]
fig, ax = plt.subplots(figsize=(8, 4.5), layout='constrained')
bars = ax.bar(shown.index, shown['per_million'])
labels = [f'{deaths:,} deaths\n{people / 1e6:.1f} M people'
          for deaths, people in zip(shown['deaths'], shown['population'])]
ax.bar_label(bars, labels=labels, fontsize=9, padding=3)
unknown = summary.loc['Unknown age', 'deaths']
ax.set(ylim=(0, shown['per_million'].max() * 1.3),
       ylabel='Road deaths per million inhabitants',
       xlabel=f'Age group. Not shown: {unknown} deaths of unknown age; '
              f'LI and ME report no age breakdown.',
       title='Road deaths by age group, 2024, 33 European countries (Eurostat)')
fig.savefig('road-deaths-by-age.png', dpi=160)
plt.show()

## Step 13 : Audit

The notebook contains an analysis in the style of an AI assistant, written by the course team. It
runs, and it has three defects :

1. Rows that are not country counts, the rates and the EU aggregate, multiply the joined cells
2. `errors='coerce'` turns the labels for babies and for people aged 100 and over into NaN, which
   removes them from the population
3. The inner join drops the deaths of unknown age and two countries, so the deaths no longer add up

Write one check per defect that fails on the proposal, without looking back at the slides. Which of
your own steps prevents each defect?

In [ ]:
raw_deaths = pd.read_csv(DATA / 'eurostat-road-deaths-2023-2024.csv').rename(
    columns={'TIME_PERIOD': 'year', 'OBS_VALUE': 'value'})
raw_population = pd.read_csv(DATA / 'eurostat-population-2024.csv').rename(
    columns={'TIME_PERIOD': 'year', 'OBS_VALUE': 'value'})
CODES = ['Y_LT15', 'Y15-17', 'Y18-24', 'Y25-49', 'Y50-64', 'Y_GE65']

rows = raw_deaths.loc[raw_deaths['year'].eq(2024) & raw_deaths['sex'].eq('T')
                      & raw_deaths['pers_cat'].eq('TOTAL') & raw_deaths['age'].ne('TOTAL')]
years = pd.to_numeric(raw_population['age'].str[1:], errors='coerce')
bands = raw_population.assign(age=pd.cut(years, BINS, right=False, labels=CODES))
bands = (bands.loc[bands['sex'].eq('T')]
         .groupby(['geo', 'age'], observed=True, as_index=False)['value'].sum())
joined = rows.merge(bands.astype({'age': 'str'}), on=['geo', 'age'],
                    suffixes=('', '_population'))
answer = joined.groupby('age').agg(deaths=('value', 'sum'),
                                   population=('value_population', 'sum'))
answer['per_million'] = (answer['deaths'] / answer['population'] * 1_000_000).round(1)
print(answer)

In [ ]:
# --- Check 1: grain. Are these country counts, one row per cell?
print(joined['unit'].unique().tolist(), 'EU27_2020' in set(joined['geo']))
print(joined.duplicated(['geo', 'age']).sum(), 'repeated cells')

# --- Check 2: ages. Which population labels did errors='coerce' turn into NaN?
lost = raw_population.loc[years.isna(), 'age'].unique().tolist()
print(lost)

# --- Check 3: reconciliation. Do all deaths and countries survive the join?
all_countries = set(raw_deaths['geo']) - {'EU27_2020'}
print(sorted(all_countries - set(joined['geo'])))
print('UNK' in set(joined['age']))

## If time permits :

1. Collect Steps 4–10 into a function `rates_for(sex)` ; the notebook gives its outline. Compute
   the rates for `'F'` and `'M'` and put the two `per_million` columns side by side. In which age
   group is the difference between women and men largest?
2. Write 6–10 sentences in the answer cell : counts and denominators, missing values and zeros,
   unknown ages, the defects of the proposal, what a rate per inhabitant cannot show, such as
   exposure and causes, and how another analyst reruns your notebook. If all 183 deaths of unknown
   age were in the 65+ group, would its rate overtake that of the 18–24 group?

In [ ]:
def rates_for(sex):
    counts = deaths.loc[deaths['unit'].eq('NR') & deaths['pers_cat'].eq('TOTAL')
                        & deaths['year'].eq(2024) & deaths['sex'].eq(sex)
                        & deaths['geo'].ne('EU27_2020') & deaths['age'].ne('TOTAL')]
    counts = counts.assign(age_group=counts['age'].map(AGE_CODES))
    counts = counts[['geo', 'age_group', 'value']].rename(columns={'value': 'deaths'})
    people = population.loc[population['sex'].eq(sex) & population['geo'].ne('EU27_2020')]
    ages = people.loc[~people['age'].isin(['TOTAL', 'UNK'])]
    years = ages['age'].replace({'Y_LT1': 'Y0', 'Y_OPEN': 'Y100'}).str[1:].astype(int)
    ages = ages.assign(age_group=pd.cut(years, BINS, right=False, labels=LABELS[:-1]))
    bands = (ages.groupby(['geo', 'age_group'], observed=True)['value'].sum()
             .reset_index(name='population'))
    bands['age_group'] = bands['age_group'].astype(str)
    cells = pd.MultiIndex.from_product([COUNTRIES, LABELS], names=['geo', 'age_group'])
    table = cells.to_frame(index=False).merge(bands, how='left', validate='one_to_one')
    table = table.merge(counts, how='left', validate='one_to_one')
    table['reported_population'] = table['population'].where(table['deaths'].notna())
    grouped = table.groupby('age_group', sort=False)
    return (grouped['deaths'].sum() / grouped['reported_population'].sum(min_count=1)
            * 1_000_000).round(1)


by_sex = pd.DataFrame({'women': rates_for('F'), 'men': rates_for('M')})
by_sex['difference'] = by_sex['men'] - by_sex['women']
print(by_sex)
print(by_sex['difference'].idxmax())

The rates divide the recorded road deaths by the population on 1 January 2024 ; both come from
Eurostat and cover the same 33 countries. Liechtenstein and Montenegro report no breakdown by
age : their cells stay missing, and each age rate uses only the population of the 31 countries
that report the group. 183 deaths have no recorded age. If all of them were aged 65 and over, the
65+ rate would rise from 62.9 to 64.7 per million, still below the 71.7 of the 18–24 group. The
proposal mixed rates and the EU aggregate into the counts, lost babies and people aged 100 and
over through `errors='coerce'`, and dropped deaths and countries in an inner join ; each check
above catches one of these defects. A rate per inhabitant does not measure exposure, such as the
distance driven, or causes. The notebook reads the unchanged Eurostat files from `datasets/`, so
another analyst reruns it with Run All and gets the same table.